# Second-request-only ablation (reviewer check)

**Question.** Does the detector use the *precondition* (first) request, or does the second request
alone carry the signal? A logistic regression on the second request alone separates the classes as
well as the pair (see manuscript, Section "Surface-feature baselines"). This notebook asks the same
question of the two neural models.

**Design.** Identical to notebooks 04 (Longformer, 2e-5) and 06 (LSTM, 1e-3): same 16 LOAO folds,
same 3 seeds (42, 43, 44), same 5 epochs, batch size 8, same tokenizer. The **only** change: every
training and evaluation sequence is reduced to one request before tokenization
(`VARIANT = "second"`: the request after `[SEP]`; `"first"`: the request before it). Because the
number of training records is unchanged, the number of update steps is unchanged (matched budget).
The single-request Spree row (no `[SEP]`) is kept as is during training, and **dropped from the
comparison** so the pooled set is the primary 546 predictions.

**Outputs** (Drive, under `thesis/results/`): `longformer_<variant>_only/` and `lstm_<variant>_only/`,
one JSON per fold and run in the same format as the originals, plus
`ablation_<variant>_summary.json`. The last cell compares against the pair results.

**Runtime.** 48 Longformer runs (about as long as notebook 04) plus 48 LSTM runs. Use an A100 and
pilot first (`PILOT_FOLDS = ["Grafana"]`).

**How to read the result.**
- If the second-only models score about as well as the pair, the scores do not show that the
  precondition request is used.
- If they score clearly worse, the first request contributes.
- Run it again with `VARIANT = "first"` as a control: a model that reads only the precondition
  request should do much worse.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, glob, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import (LongformerForSequenceClassification, LongformerTokenizer,
                          TrainingArguments, Trainer)
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------ configuration ------------------------------
BASE = "/content/drive/MyDrive/thesis"
PRETRAINED_PATH = f"{BASE}/models/longformer/phase3_final"   # tokenizer + Longformer weights
FOLDS_DIR = f"{BASE}/data/loao_folds_v2"                      # the restored July folds

VARIANT = "second"        # "second" = ablation, "first" = control
PILOT_FOLDS = None        # e.g. ["Grafana"] to pilot; None = all 16 folds
N_RUNS = 3
MAX_LENGTH = 256
SEP = " [SEP] "

LF_DIR   = f"{BASE}/results/longformer_{VARIANT}_only"
LSTM_DIR = f"{BASE}/results/lstm_{VARIANT}_only"
REF_LF_DIR   = f"{BASE}/results/longformer_finetune_v2"
REF_LSTM_DIR = f"{BASE}/results/lstm_comparator"
for d in (LF_DIR, LSTM_DIR):
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = LongformerTokenizer.from_pretrained(PRETRAINED_PATH)
print("device:", device, "| variant:", VARIANT)

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def reduce_sequence(seq, variant=VARIANT):
    # Keep one request. Rows without [SEP] (the single-step Spree exemplar) are left unchanged.
    parts = seq.split(SEP)
    if len(parts) < 2:
        return seq
    return parts[-1] if variant == "second" else parts[0]

def load_reduced(path):
    recs = load_jsonl(path)
    return [{**r, "sequence": reduce_sequence(r["sequence"])} for r in recs]

def fold_names():
    names = sorted(d.replace("fold_", "") for d in os.listdir(FOLDS_DIR) if d.startswith("fold_"))
    return PILOT_FOLDS if PILOT_FOLDS else names

# quick sanity check on one fold: show what the models will see
_f = fold_names()[0]
_orig = load_jsonl(f"{FOLDS_DIR}/fold_{_f}/eval.jsonl")[:2]
for r in _orig:
    print("label", r["label"])
    print("  pair   :", r["sequence"][:140])
    print("  reduced:", reduce_sequence(r["sequence"])[:140])

In [ ]:
# ============================ Longformer (as notebook 04) ============================
class BinaryClassDataset(Dataset):
    def __init__(self, records, tokenizer, max_length):
        self.sequences = [r["sequence"] for r in records]
        self.labels = [r["label"] for r in records]
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.sequences[idx], truncation=True, max_length=self.max_length, return_tensors="pt")
        ids, am = enc["input_ids"][0], enc["attention_mask"][0]
        gam = torch.zeros_like(am); gam[0] = 1          # global attention on the first token
        return {"input_ids": ids, "attention_mask": am, "global_attention_mask": gam,
                "labels": torch.tensor(self.labels[idx], dtype=torch.long)}

def lf_collate(batch):
    max_len = max(b["input_ids"].size(0) for b in batch)
    def pad(t, v):
        n = max_len - t.size(0)
        return t if n == 0 else torch.cat([t, torch.full((n,), v, dtype=t.dtype)])
    return {"input_ids": torch.stack([pad(b["input_ids"], tokenizer.pad_token_id) for b in batch]),
            "attention_mask": torch.stack([pad(b["attention_mask"], 0) for b in batch]),
            "global_attention_mask": torch.stack([pad(b["global_attention_mask"], 0) for b in batch]),
            "labels": torch.stack([b["labels"] for b in batch])}

def longformer_fold(fold, run_idx, seed):
    random.seed(seed); torch.manual_seed(seed)
    tr = load_reduced(f"{FOLDS_DIR}/fold_{fold}/train.jsonl")
    ev = load_reduced(f"{FOLDS_DIR}/fold_{fold}/eval.jsonl")
    train_ds, eval_ds = BinaryClassDataset(tr, tokenizer, MAX_LENGTH), BinaryClassDataset(ev, tokenizer, MAX_LENGTH)
    model = LongformerForSequenceClassification.from_pretrained(PRETRAINED_PATH, num_labels=2)
    args = TrainingArguments(output_dir=f"/tmp/ft_{VARIANT}_{fold}_{run_idx}", num_train_epochs=5,
        per_device_train_batch_size=8, gradient_accumulation_steps=1, per_device_eval_batch_size=8,
        learning_rate=2e-5, logging_steps=10, save_strategy="no", eval_strategy="no",
        report_to="none", fp16=torch.cuda.is_available(), seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=lf_collate)
    trainer.train()
    model.eval(); probs, labels = [], []
    with torch.no_grad():
        for i in range(len(eval_ds)):
            item = lf_collate([eval_ds[i]])
            item = {k: v.to(model.device) for k, v in item.items() if k != "labels"}
            probs.append(torch.softmax(model(**item).logits, dim=-1)[0, 1].item())
            labels.append(eval_ds.labels[i])
    out = {"fold": fold, "run": run_idx, "seed": seed, "variant": VARIANT, "probs": probs,
           "labels": labels, "n_train": len(train_ds), "n_eval": len(eval_ds)}
    json.dump(out, open(f"{LF_DIR}/fold_{fold}_run{run_idx}.json", "w"))
    del model, trainer; torch.cuda.empty_cache()
    return out

In [ ]:
# ============================ LSTM (as notebook 06) ============================
class SeqDataset(Dataset):
    def __init__(self, records, tokenizer, max_length):
        self.sequences = [r["sequence"] for r in records]
        self.labels = [r["label"] for r in records]
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx):
        ids = self.tokenizer(self.sequences[idx], truncation=True, max_length=self.max_length)["input_ids"]
        return torch.tensor(ids, dtype=torch.long), self.labels[idx]

def lstm_collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    padded = pad_sequence(seqs, batch_first=True, padding_value=tokenizer.pad_token_id)
    return padded, lengths, torch.tensor(labels, dtype=torch.long)

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=tokenizer.pad_token_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, 2)
    def forward(self, input_ids, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(self.embedding(input_ids), lengths.cpu(),
                                                   batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        return self.classifier(self.dropout(torch.cat([h_n[-2], h_n[-1]], dim=-1)))

def lstm_fold(fold, run_idx, seed, epochs=5, batch_size=8, lr=1e-3):
    random.seed(seed); torch.manual_seed(seed); np.random.seed(seed)
    tr = load_reduced(f"{FOLDS_DIR}/fold_{fold}/train.jsonl")
    ev = load_reduced(f"{FOLDS_DIR}/fold_{fold}/eval.jsonl")
    train_ds, eval_ds = SeqDataset(tr, tokenizer, MAX_LENGTH), SeqDataset(ev, tokenizer, MAX_LENGTH)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=lstm_collate)
    model = LSTMClassifier(len(tokenizer)).to(device)
    opt, crit = torch.optim.Adam(model.parameters(), lr=lr), nn.CrossEntropyLoss()
    model.train(); step = 0
    for _ in range(epochs):
        for ids, lengths, labels in loader:
            opt.zero_grad()
            loss = crit(model(ids.to(device), lengths), labels.to(device))
            loss.backward(); opt.step(); step += 1
    model.eval(); probs, labels = [], []
    with torch.no_grad():
        for i in range(len(eval_ds)):
            ids, lab = eval_ds[i]
            ids = ids.unsqueeze(0).to(device)
            logits = model(ids, torch.tensor([ids.size(1)]))
            probs.append(torch.softmax(logits, dim=-1)[0, 1].item()); labels.append(lab)
    out = {"fold": fold, "run": run_idx, "seed": seed, "variant": VARIANT, "probs": probs,
           "labels": labels, "n_train": len(train_ds), "n_eval": len(eval_ds), "total_steps": step}
    json.dump(out, open(f"{LSTM_DIR}/fold_{fold}_run{run_idx}.json", "w"))
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return out

In [ ]:
# ============================ run (resumable: skips finished fold/run files) ============================
for fold in fold_names():
    for run_idx in range(N_RUNS):
        seed = 42 + run_idx
        for name, fn, d in (("longformer", longformer_fold, LF_DIR), ("lstm", lstm_fold, LSTM_DIR)):
            path = f"{d}/fold_{fold}_run{run_idx}.json"
            if os.path.exists(path):
                continue
            r = fn(fold, run_idx, seed)
            print(f"{name:10s} {fold:22s} run{run_idx} seed={seed}  "
                  f"pos={sum(r['labels'])} neg={len(r['labels'])-sum(r['labels'])}  "
                  f"mean P(attack)={np.mean(r['probs']):.3f}")
print("done")

In [ ]:
# ============================ comparison with the pair models ============================
def load_dir(d):
    out = {}
    for f in sorted(glob.glob(f"{d}/fold_*_run*.json")):
        r = json.load(open(f)); out[(r["fold"], r["run"])] = r
    return out

def primary_mask(fold):
    # primary evaluation rows = those with two requests; drops the single-step Spree row
    return np.array([SEP in r["sequence"] for r in load_jsonl(f"{FOLDS_DIR}/fold_{fold}/eval.jsonl")])

def pooled(dir_):
    res = load_dir(dir_)
    P, L, keys = [], [], []
    for (fold, run), r in sorted(res.items()):
        m = primary_mask(fold)
        assert len(m) == len(r["probs"]), (fold, run, len(m), len(r["probs"]))
        P += list(np.array(r["probs"])[m]); L += list(np.array(r["labels"])[m]); keys += [(fold, run)] * int(m.sum())
    return np.array(P), np.array(L), keys

def metrics(p, y, thr=0.5):
    pred = p >= thr
    tp, fp = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum())
    fn, tn = int((~pred & (y == 1)).sum()), int((~pred & (y == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    return {"n": len(y), "pos": int(y.sum()), "recall": rec, "precision": prec,
            "fpr": fp / (fp + tn) if fp + tn else 0.0,
            "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
            "roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p)}

models = {"Longformer, pair": REF_LF_DIR, "LSTM, pair": REF_LSTM_DIR,
          f"Longformer, {VARIANT} only": LF_DIR, f"LSTM, {VARIANT} only": LSTM_DIR}
pool = {k: pooled(v) for k, v in models.items()}
summary = {k: metrics(p, y) for k, (p, y, _) in pool.items()}

print(f"{'model':28s} {'n':>4s} {'pos':>4s} {'recall':>7s} {'prec':>7s} {'FPR':>7s} {'F1':>7s} {'ROC':>7s} {'PR':>7s}")
for k, m in summary.items():
    print(f"{k:28s} {m['n']:4d} {m['pos']:4d} {m['recall']:7.3f} {m['precision']:7.3f} {m['fpr']:7.3f} "
          f"{m['f1']:7.3f} {m['roc_auc']:7.3f} {m['pr_auc']:7.3f}")
print("\nNote: the paper's pooled numbers (Longformer recall .646 F1 .546 | LSTM recall .823 F1 .756) come from"
      "\nthe released pooled_predictions.npz. The Spree fold's per-run files were re-scored after the fold was"
      "\namended with the single-step case, so pair rows computed here differ slightly from the paper"
      "\n(about 0.01-0.03) and only in the Spree fold. The pair-versus-single-request comparison below uses"
      "\nthe same re-scored fold on both sides, so it is internally consistent.")

# Paired bootstrap of F1 (pair minus single-request), within-pool, as in the manuscript
rng = np.random.default_rng(0)
def f1_of(p, y, thr=0.5):
    pred = p >= thr
    tp = (pred & (y == 1)).sum(); fp = (pred & (y == 0)).sum(); fn = (~pred & (y == 1)).sum()
    return 2 * tp / (2 * tp + fp + fn) if tp else 0.0
boot = {}
for arch in ("Longformer", "LSTM"):
    pp, yp, kp = pool[f"{arch}, pair"]
    pa, ya, ka = pool[f"{arch}, {VARIANT} only"]
    assert kp == ka and (yp == ya).all(), "rows are not aligned between pair and ablation"
    diffs = []
    for _ in range(10000):
        idx = rng.integers(0, len(yp), len(yp))
        diffs.append(f1_of(pp[idx], yp[idx]) - f1_of(pa[idx], ya[idx]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    boot[arch] = {"f1_pair_minus_single": f1_of(pp, yp) - f1_of(pa, ya), "ci95": [lo, hi]}
    print(f"{arch}: F1(pair) - F1({VARIANT} only) = {boot[arch]['f1_pair_minus_single']:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")

# Per-fold recall for the single-request models
print("\nPer-fold recall (mean over runs):")
print(f"{'fold':24s} {'LF pair':>8s} {'LF '+VARIANT:>10s} {'LSTM pair':>10s} {'LSTM '+VARIANT:>12s}")
for fold in fold_names():
    row = []
    for k in models:
        p, y, keys = pool[k]
        sel = [i for i, kk in enumerate(keys) if kk[0] == fold and y[i] == 1]
        row.append(np.mean(p[sel] >= 0.5) if sel else float("nan"))
    print(f"{fold:24s} {row[0]:8.3f} {row[2]:10.3f} {row[1]:10.3f} {row[3]:12.3f}")

json.dump({"variant": VARIANT, "pooled": summary, "bootstrap": boot},
          open(f"{BASE}/results/ablation_{VARIANT}_summary.json", "w"), indent=1)
print(f"\nsaved {BASE}/results/ablation_{VARIANT}_summary.json")